In [38]:
import pandas as pd
import numpy as np
import re
upi = pd.read_csv("../data/raw/track1_upi_transactions.csv")
upi.isnull().sum()

txn_id            0
timestamp         0
user_id           0
merchant_id       0
amount            0
utr            1024
mcc            2926
status            0
dtype: int64

In [39]:
# Column: txn_id is clean (excepty having duplicates)

# 1. Check for null values
has_nulls = upi['txn_id'].isna().any()

# 2. Check the exact format using a strict pattern
# ^TXN   -> Must start exactly with 'TXN'
# \d{8}$ -> Must end with exactly 8 digits (which automatically blocks spaces and symbols)
is_valid_format = upi['txn_id'].astype(str).str.fullmatch(r'^TXN\d{8}$')

# 3. Isolate any rows that fail the checks
bad_ids_mask = upi['txn_id'].isna() | ~is_valid_format
invalid_txns = upi[bad_ids_mask]

# 4. Print the validation report
print(f"Contains Null Values: {has_nulls}")
print(f"Total Malformed TXN IDs: {len(invalid_txns)} / {len(upi)}")

if not invalid_txns.empty:
    print("\n🚨 FRAUD/DATA ALERT: Here are the IDs that failed the formatting check:")
    print(invalid_txns['txn_id'].head(15))
else:
    print("\n✅ All TXN IDs are perfectly clean and follow the strict TXN00000000 format!")

Contains Null Values: False
Total Malformed TXN IDs: 0 / 20400

✅ All TXN IDs are perfectly clean and follow the strict TXN00000000 format!


In [40]:
# Column: timestamp - no date only nor null values present

# 1. Clean up the raw strings
raw_time = upi['timestamp'].astype(str).str.strip()
raw_time = raw_time.replace(['nan', 'None', 'null', ''], np.nan)

# 2. Identify the structure of the raw data
is_unix = raw_time.str.match(r'^\d{10}$', na=False)  # e.g., 1770063471
has_colon = raw_time.str.contains(':', na=False)     # e.g., 2026-01-15 00:11:30

# 3. Parse Unix timestamps and standard string timestamps natively
unix_t = pd.to_datetime(pd.to_numeric(raw_time.where(is_unix), errors='coerce'), unit='s', errors='coerce')
str_t = pd.to_datetime(raw_time.where(~is_unix), format='mixed', errors='coerce')

# Combine them into a native, math-ready datetime column
upi['timestamp_clean'] = unix_t.combine_first(str_t)

# 4. Generate the SINGLE Categorical Status Column
conditions = [
    upi['timestamp_clean'].isna(),                             # Condition 1: Missing entirely
    upi['timestamp_clean'].notna() & ~is_unix & ~has_colon     # Condition 2: Parsed, but no time provided
]
choices = ['MISSING', 'DATE_ONLY']

# Apply conditions, defaulting to 'VALID' if it passes both checks
upi['timestamp_status'] = np.select(conditions, choices, default='VALID')

# 5. Finalize columns
upi['timestamp'] = upi['timestamp_clean']
upi = upi.drop(columns=['timestamp_clean'])

# Check results
print(upi['timestamp_status'].value_counts())

timestamp_status
VALID        19388
DATE_ONLY     1012
Name: count, dtype: int64


In [41]:
# Column: utr

# 1. Force to string, uppercase, and remove all spaces 
# (This perfectly fixes entries like 'UTR 2787678319')
upi['utr_clean'] = upi['utr'].astype(str).str.upper().str.replace(' ', '')

# 2. Convert text-based nulls back into actual Pandas missing values
upi['utr_clean'] = upi['utr_clean'].replace(['NAN', 'NONE', 'NULL', ''], np.nan)

# 3. Create the Missing UTR Flag
upi['is_missing_utr'] = upi['utr_clean'].isna()

# 4. Finalize columns (Skipping duplicate checks for now!)
upi['utr'] = upi['utr_clean']
upi = upi.drop(columns=['utr_clean'])

# Check results
print(f"Total Missing UTRs: {upi['is_missing_utr'].sum()}")
# print(upi[upi['is_missing_utr']==True])

Total Missing UTRs: 1024


In [42]:
# Column: user_id

# 1. Check for missing/null values
has_nulls = upi['user_id'].isna().any()

# 2. Check the exact format using a strict pattern
# ^USR  -> Must start exactly with 'USR'
# \d+$  -> Must end with one or more digits (blocks spaces and special characters)
is_valid_format = upi['user_id'].astype(str).str.fullmatch(r'^USR\d+$')

# 3. Isolate any rows that fail the strict checks
bad_ids_mask = upi['user_id'].isna() | ~is_valid_format
invalid_users = upi[bad_ids_mask]

# 4. Print the validation report
print(f"Contains Null Values: {has_nulls}")
print(f"Total Malformed User IDs: {len(invalid_users)} / {len(upi)}")

if not invalid_users.empty:
    print("\n🚨 DATA ALERT: Here are the User IDs that failed the formatting check:")
    print(invalid_users['user_id'].head(15))
else:
    print("\n✅ All User IDs are perfectly clean and follow the strict USR + digits format!")

Contains Null Values: False
Total Malformed User IDs: 0 / 20400

✅ All User IDs are perfectly clean and follow the strict USR + digits format!


In [43]:
# Column: merchant_ids

# 1. Check for missing/null values
has_nulls = upi['merchant_id'].isna().any()

# 2. Check the exact format using a strict pattern
# ^MCH  -> Must start exactly with 'MCH'
# \d+$  -> Must end with one or more digits (blocks spaces and symbols)
is_valid_format = upi['merchant_id'].astype(str).str.fullmatch(r'^MCH\d+$')

# 3. Isolate any rows that fail the strict checks
bad_ids_mask = upi['merchant_id'].isna() | ~is_valid_format
invalid_merchants = upi[bad_ids_mask]

# 4. Print the validation report
print(f"Contains Null Values: {has_nulls}")
print(f"Total Malformed Merchant IDs: {len(invalid_merchants)} / {len(upi)}")

if not invalid_merchants.empty:
    print("\n🚨 DATA ALERT: Here are the Merchant IDs that failed the formatting check:")
    print(invalid_merchants['merchant_id'].head(15))
else:
    print("\n✅ All Merchant IDs are perfectly clean and follow the strict MCH + digits format!")

Contains Null Values: False
Total Malformed Merchant IDs: 0 / 20400

✅ All Merchant IDs are perfectly clean and follow the strict MCH + digits format!


In [44]:
# Column: amounts

def clean_upi_amount(value):
    # Catches actual programmatic nulls (np.nan, None)
    if pd.isnull(value):
        return None
    
    # Convert to lowercase and strip extra spaces on the edges
    v = str(value).lower().strip()
    
    # THE NULL CATCHER: Intercept text-based empty values
    if v in ['not available', 'na', 'n/a', 'nan', 'null', 'missing', 'none', '-', '']:
        return None
        
    # Erase symbols, spaces, commas AND currency words
    v = re.sub(r'[₹$,\s]|inr|rs\.?|rupees?', '', v)
    
    multiplier = 1
    
    # Check for the value suffixes
    if v.endswith('k'):
        multiplier = 1_000
        v = v.replace('k', '')
    elif v.endswith('l') or v.endswith('lakh'):
        multiplier = 100_000
        v = re.sub(r'(l|lakh)$', '', v)
    elif v.endswith('cr') or v.endswith('crore'):
        multiplier = 10_000_000
        v = re.sub(r'(cr|crore)$', '', v)
    elif v.endswith('m'):
        multiplier = 1_000_000
        v = v.replace('m', '')

    # Convert and multiply
    try:
        return float(v) * multiplier
    except ValueError:
        return None

# Apply the cleaning function to create a temporary clean column
upi['amount_clean'] = upi['amount'].apply(clean_upi_amount)

# Define conditions to categorize amounts into a single status column
conditions = [
    upi['amount_clean'].isna(),        # Missing or invalid amount
    upi['amount_clean'] < 0,           # Negative amount (Refunds/Errors)
    upi['amount_clean'] == 0,          # Zero-value transactions
    upi['amount_clean'] > 50000        # High-value transactions (> 50,000)
]
choices = ['MISSING', 'NEGATIVE', 'ZERO', 'HIGH_AMOUNT']

# Generate the single categorical status column
upi['amount_status'] = np.select(conditions, choices, default='VALID')

# Replace the original column and clean up the temporary column
upi['amount'] = upi['amount_clean']
upi = upi.drop(columns=['amount_clean'])

# Report breakdown
print(upi['amount_status'].value_counts(dropna=False))
# print("\nSample of Flagged Non-Standard Amounts:")
# print(upi[upi['amount_status'] != 'VALID'][['amount', 'amount_status']].head(10))

amount_status
VALID       19971
NEGATIVE      429
Name: count, dtype: int64


In [45]:
# Column: utr

# 1. Force to string, uppercase, and remove all spaces 
upi['utr_clean'] = upi['utr'].astype(str).str.upper().str.replace(' ', '')

# 2. Convert text-based nulls to actual Pandas missing values
upi['utr_clean'] = upi['utr_clean'].replace(['NAN', 'NONE', 'NULL', ''], np.nan)

# 3. Define conditions for the single categorical status column
conditions = [
    upi['utr_clean'].isna(),                                               # Condition 1: Missing entirely
    upi.duplicated(subset=['utr_clean'], keep=False) & upi['utr_clean'].notna() # Condition 2: Duplicate UTR (Double-spend)
]
choices = ['MISSING', 'DUPLICATE']

# Apply conditions, defaulting to 'VALID' if it passes both checks
upi['utr_status'] = np.select(conditions, choices, default='VALID')

# 4. Finalize columns
upi['utr'] = upi['utr_clean']
upi = upi.drop(columns=['utr_clean', 'is_missing_utr'])

print(upi['utr_status'].value_counts())

utr_status
VALID        18624
MISSING       1024
DUPLICATE      752
Name: count, dtype: int64


In [46]:
# Column: mcc
# null values will be corrected in pipeline

# 1. Force to string, strip spaces, and clean up text nulls
mcc_raw = upi['mcc'].astype(str).str.strip()
mcc_raw = mcc_raw.replace(['nan', 'None', 'null', '', 'nat'], np.nan)

# 2. Extract digits only and format as clean 4-digit string codes (or numeric if preferred)
upi['mcc_clean'] = mcc_raw.str.extract(r'(\d{4})')[0]

# 3. Create a single categorical status column for MCC data quality
conditions = [
    upi['mcc_clean'].isna(),  # Missing MCC
]
choices = ['MISSING']

upi['mcc_status'] = np.select(conditions, choices, default='VALID')

# 4. Finalize columns
upi['mcc'] = upi['mcc_clean']
upi = upi.drop(columns=['mcc_clean'])

print(upi['mcc_status'].value_counts())
print(f"\nTotal Missing MCCs: {upi['mcc'].isna().sum()}")
print(upi[['mcc', 'mcc_status']].head(10))

mcc_status
VALID      17474
MISSING     2926
Name: count, dtype: int64

Total Missing MCCs: 2926
    mcc mcc_status
0  5411      VALID
1  4131      VALID
2  5411      VALID
3  5411      VALID
4  4131      VALID
5  5812      VALID
6   NaN    MISSING
7  5411      VALID
8   NaN    MISSING
9  5411      VALID


In [47]:
# Column: status

status_mapping = {
    'COMPLETED': 'SUCCESS',
    'TXN_FAILED': 'FAILED',
    'S': 'SUCCESS',
    'TXN_SUCCESS': 'SUCCESS',
    'Success': 'SUCCESS',
    'Initiated': 'PROCESSING',
    'Pending': 'PROCESSING',
    'Fail': 'FAILED',
    'Declined': 'FAILED',
    'F': 'FAILED',
    'PENDING': 'PROCESSING',
}
upi['status'] = upi['status'].replace(status_mapping)
upi['status'].unique()

array(['SUCCESS', 'FAILED', 'PROCESSING'], dtype=object)

In [48]:
upi = upi[[
    'txn_id', 
    'timestamp', 
    'timestamp_status',
    'user_id', 
    'merchant_id', 
    'amount', 
    'amount_status',
    'utr', 
    'utr_status',
    'mcc', 
    'mcc_status',
    'status', 
]]

In [49]:
upi.to_csv("../data/cleaned/modified_upi_transactions.csv", index=False)